In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd /data/lerobot_datasets/merged_dataset/meta/
 curl -sL -o /data/lerobot_datasets/merged_dataset/meta/$MODALITY_JSON \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_JSON"
ls -la /data/lerobot_datasets/merged_dataset/meta/
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd /data/lerobot_datasets/merged_dataset/meta/
 curl -sL -o /data/lerobot_datasets/merged_dataset/meta/$MODALITY_PY \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_PY"
ls -la /data/lerobot_datasets/merged_dataset/meta/
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sed -i 's/eval_strategy: str = "no"/eval_strategy: str = "steps"/' $HOME/Isaac-GR00T/gr00t/configs/training/training_config.py
sed -i 's/# \(compute_metrics=partial(compute_eval_accuracy, action_offset=self\.action_offset),\)/\1/' $HOME/Isaac-GR00T/gr00t/experiment/trainer.py
echo "=== training_config.py eval_strategy ==="
grep "eval_strategy" $HOME/Isaac-GR00T/gr00t/configs/training/training_config.py
echo "=== trainer.py compute_metrics ==="
grep "compute_metrics" $HOME/Isaac-GR00T/gr00t/experiment/trainer.py
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export WANDB_API_KEY=$WANDB_API_KEY
uv run wandb login
cd \$HOME/Isaac-GR00T
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T
rm -f \$HOME/TRAINING_DONE \$HOME/TRAINING_FAILED
tmux kill-session -t finetune 2>/dev/null || true
tmux new-session -d -s finetune
tmux send-keys -t finetune 'export WANDB_API_KEY=$WANDB_API_KEY' C-m
tmux send-keys -t finetune 'export WANDB_NAME=gr00t-n1d6-so100' C-m
tmux send-keys -t finetune 'export PATH=\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin' C-m
tmux send-keys -t finetune 'cd \$HOME/Isaac-GR00T && CUDA_VISIBLE_DEVICES=0 uv run python gr00t/experiment/launch_finetune.py --base_model_path nvidia/GR00T-N1.7-3B --dataset_path /data/lerobot_datasets/merged_dataset --modality_config_path /data/lerobot_datasets/merged_dataset/meta/$MODALITY_PY --embodiment_tag NEW_EMBODIMENT --num_gpus 1 --output_dir ./finetuned_models/$DATASET_NAME/ --save_steps $SAVE_STEPS --save_total_limit 5 --max_steps $MAX_STEPS --use-wandb --warmup_ratio 0.05 --weight_decay 1e-5 --learning_rate 1e-4 --global_batch_size 128 --color_jitter_params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 --dataloader_num_workers 20 2>&1 | tee \$HOME/finetune.log && touch \$HOME/TRAINING_DONE || touch \$HOME/TRAINING_FAILED' C-m
tmux send-keys -t finetune 'exit' C-m
EOF

In [0]:
%sh
echo "Waiting for training to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t finetune 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Training complete!"
    break
  fi
  
  sleep 60
done

echo "Cleaning up finetuned model files..."

ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T/finetuned_models/$DATASET_NAME/
du -sh .
EOF

In [0]:
%sh
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  $SSH_USER@$BREV_IP:/home/SSH_USER/Isaac-GR00T/finetuned_models/ \
  /Volumes/workspace/default/trained_models/

In [0]:
# %sh
# ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
# sudo shutdown -h now
# EOF
